In [1]:
import random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [2]:
from datasets import load_dataset

ner_dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

print(ner_dataset)
print(ner_dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [3]:
# =========================
# QUESTION 2 — NAMED ENTITY RECOGNITION
# CoNLL-2003: CRF vs BERT
# =========================

# 1. Install libraries
!pip install -q "datasets<4.0.0" seqeval sklearn-crfsuite transformers evaluate accelerate

# 2. Imports and seed
import random
import numpy as np
import torch
import pandas as pd

from datasets import load_dataset
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
import sklearn_crfsuite

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

import evaluate

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# =========================
# 3. Load CoNLL-2003 dataset
# =========================

ner_dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

print("\nDataset structure:")
print(ner_dataset)

print("\nFirst training example:")
print(ner_dataset["train"][0])

# =========================
# 4. Label names and BIO structure
# =========================

label_names = ner_dataset["train"].features["ner_tags"].feature.names

print("\nNER labels:")
print(label_names)
print("Number of labels:", len(label_names))

print("\nBIO label explanation:")
print("O      = Outside any named entity")
print("B-PER  = Beginning of a person entity")
print("I-PER  = Inside a person entity")
print("B-ORG  = Beginning of an organization entity")
print("I-ORG  = Inside an organization entity")
print("B-LOC  = Beginning of a location entity")
print("I-LOC  = Inside a location entity")
print("B-MISC = Beginning of a miscellaneous entity")
print("I-MISC = Inside a miscellaneous entity")

example = ner_dataset["train"][0]
tokens = example["tokens"]
ner_tags = example["ner_tags"]
ner_labels = [label_names[tag] for tag in ner_tags]

print("\nFirst example with token-label pairs:")
for token, label in zip(tokens, ner_labels):
    print(token, "→", label)

# =========================
# 5. Create smaller subsets
# =========================

train_data = ner_dataset["train"].shuffle(seed=SEED).select(range(3000))
val_data = ner_dataset["validation"].shuffle(seed=SEED).select(range(800))

print("\nSubset sizes:")
print("Train size:", len(train_data))
print("Validation size:", len(val_data))

# ======================================================
# PART A — CRF MODEL
# ======================================================

# =========================
# 6. CRF feature extraction
# =========================

def word2features(sent, i):
    word = sent[i]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    if i > 0:
        prev_word = sent[i - 1]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        next_word = sent[i + 1]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(tag_ids):
    return [label_names[tag_id] for tag_id in tag_ids]


X_train_crf = [sent2features(example["tokens"]) for example in train_data]
y_train_crf = [sent2labels(example["ner_tags"]) for example in train_data]

X_val_crf = [sent2features(example["tokens"]) for example in val_data]
y_val_crf = [sent2labels(example["ner_tags"]) for example in val_data]

print("\nExample CRF features for first token:")
print(X_train_crf[0][0])

print("\nExample CRF labels:")
print(y_train_crf[0])

# =========================
# 7. Train CRF
# =========================

crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("\nTraining CRF model...")
crf_model.fit(X_train_crf, y_train_crf)

# =========================
# 8. Evaluate CRF
# =========================

crf_preds = crf_model.predict(X_val_crf)

crf_precision = precision_score(y_val_crf, crf_preds)
crf_recall = recall_score(y_val_crf, crf_preds)
crf_f1 = f1_score(y_val_crf, crf_preds)

print("\n=========================")
print("CRF RESULTS")
print("=========================")
print("Precision:", crf_precision)
print("Recall:", crf_recall)
print("F1-score:", crf_f1)
print()
print(classification_report(y_val_crf, crf_preds))

# ======================================================
# PART B — BERT TOKEN CLASSIFICATION
# ======================================================

# =========================
# 9. Load tokenizer
# =========================

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# =========================
# 10. Token-label alignment
# =========================

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


print("\nTokenizing and aligning labels for BERT...")
tokenized_train = train_data.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_data.map(tokenize_and_align_labels, batched=True)

print("\nTokenized example keys:")
print(tokenized_train[0].keys())

# =========================
# 11. Prepare BERT model
# =========================

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

bert_ner_model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

# =========================
# 12. Metric function
# =========================

seqeval_metric = evaluate.load("seqeval")

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                current_preds.append(label_names[pred_id])
                current_labels.append(label_names[label_id])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# =========================
# 13. Train BERT NER model
# =========================

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./bert_ner_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=bert_ner_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_ner_metrics
)

print("\nTraining BERT token classification model...")
trainer.train()

# =========================
# 14. Evaluate BERT
# =========================

bert_results = trainer.evaluate()

print("\n=========================")
print("BERT NER RESULTS")
print("=========================")
print(bert_results)

# ======================================================
# PART C — ERROR ANALYSIS
# ======================================================

# =========================
# 15. Get BERT predictions
# =========================

bert_predictions = trainer.predict(tokenized_val)

bert_logits = bert_predictions.predictions
bert_label_ids = bert_predictions.label_ids
bert_pred_ids = np.argmax(bert_logits, axis=2)

true_predictions = []
true_labels = []

for prediction, label in zip(bert_pred_ids, bert_label_ids):
    current_preds = []
    current_labels = []

    for pred_id, label_id in zip(prediction, label):
        if label_id != -100:
            current_preds.append(label_names[pred_id])
            current_labels.append(label_names[label_id])

    true_predictions.append(current_preds)
    true_labels.append(current_labels)

# =========================
# 16. Show misclassified examples
# =========================

misclassified_ner_examples = []

for idx in range(len(val_data)):
    tokens = val_data[idx]["tokens"]
    gold_labels = [label_names[tag] for tag in val_data[idx]["ner_tags"]]
    pred_labels = true_predictions[idx]

    if len(tokens) != len(pred_labels):
        continue

    errors = []

    for token, gold, pred in zip(tokens, gold_labels, pred_labels):
        if gold != pred:
            errors.append({
                "token": token,
                "gold": gold,
                "predicted": pred
            })

    if len(errors) > 0:
        misclassified_ner_examples.append({
            "tokens": tokens,
            "gold_labels": gold_labels,
            "predicted_labels": pred_labels,
            "errors": errors
        })

print("\n=========================")
print("MISCLASSIFIED NER EXAMPLES")
print("=========================")

for i, example in enumerate(misclassified_ner_examples[:5]):
    print("=" * 80)
    print("Example", i + 1)
    print("Sentence:", " ".join(example["tokens"]))
    print()

    print("Token-level errors:")
    for error in example["errors"]:
        print(
            error["token"],
            "| Gold:",
            error["gold"],
            "| Predicted:",
            error["predicted"]
        )

# ======================================================
# PART D — FINAL RESULTS TABLE
# ======================================================

results_q2 = {
    "Model": ["CRF", "BERT"],
    "Precision": [crf_precision, bert_results["eval_precision"]],
    "Recall": [crf_recall, bert_results["eval_recall"]],
    "F1-score": [crf_f1, bert_results["eval_f1"]]
}

results_df_q2 = pd.DataFrame(results_q2)

print("\n=========================")
print("FINAL Q2 RESULTS TABLE")
print("=========================")
print(results_df_q2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.9 MB/s eta 0:00:00
Device: cuda

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

First training example:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22,

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca


Training BERT token classification model...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.134678,0.091300,0.840956,0.855556,0.848193,0.973976
2,0.061779,0.074212,0.869802,0.886111,0.877881,0.978922



BERT NER RESULTS
{'eval_loss': 0.07421162724494934, 'eval_precision': 0.8698023176550784, 'eval_recall': 0.8861111111111111, 'eval_f1': 0.8778809769521844, 'eval_accuracy': 0.9789217673287394, 'eval_runtime': 0.8018, 'eval_samples_per_second': 997.791, 'eval_steps_per_second': 62.362, 'epoch': 2.0}

MISCLASSIFIED NER EXAMPLES
Example 1
Sentence: Pro-Moscow leaders in Chechnya have criticised Tim Guldimann , the Swiss diplomat who heads the OSCE Chechnya mission , saying he was biased toward Zelimkhan Yandarbiyev , president of the self-declared separatist government .

Token-level errors:
Pro-Moscow | Gold: B-MISC | Predicted: O
Chechnya | Gold: B-LOC | Predicted: B-MISC
Example 2
Sentence: Singapore hanged a Thai farmer at Changi Prison on Friday for drug trafficking , the Central Narcotics Bureau ( CNB ) said .

Token-level errors:
Prison | Gold: I-LOC | Predicted: O
Example 3
Sentence: Bristol : Gloucestershire 183 and 185-6 ( J. Russell 56 not out ) , Northamptonshire 190 ( K. Cur